### Load Checkpoint

In [1]:
from pathlib import Path
import torch

project = Path.cwd().parent

checkpoint = torch.load(
    project /
    "models" /
    "rgcn_complete_checkpoint.pt",
    map_location="cpu"
)

print(checkpoint.keys())

dict_keys(['model_state_dict', 'predictor_state_dict', 'node_embeddings', 'node2id', 'relation2id'])


### Recover Data

In [2]:
node_embeddings = checkpoint["node_embeddings"]

node2id = checkpoint["node2id"]

relation2id = checkpoint["relation2id"]

print(node_embeddings.shape)

torch.Size([7502, 128])


### Import

In [5]:
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### Load Edge_index and edge_type

In [9]:
edge_index = torch.tensor(
    np.load(project / "data" / "ml_ready" / "edge_index.npy"),
    dtype=torch.long
)

edge_type = torch.tensor(
    np.load(project / "data" / "ml_ready" / "edge_type.npy"),
    dtype=torch.long
)

num_nodes = node_embeddings.shape[0]

print(edge_index.shape)
print(edge_type.shape)
print(num_nodes)

torch.Size([2, 15573])
torch.Size([15573])
7502


### Find All CVEs

In [10]:
cve_nodes = [
    node for node in node2id.keys()
    if str(node).startswith("CVE-")
]

print("Total CVEs:", len(cve_nodes))

Total CVEs: 2024


### Select Hidden CVEs (~20%)

In [11]:
import random

random.seed(42)

num_hidden_cves = int(0.2 * len(cve_nodes))

hidden_cves = set(
    random.sample(cve_nodes, num_hidden_cves)
)

hidden_cve_ids = set(
    node2id[c] for c in hidden_cves
)

print("Hidden CVEs:", len(hidden_cves))

Hidden CVEs: 404


### Build Hidden Edge Mask

In [12]:
src_nodes_all = edge_index[0]
dst_nodes_all = edge_index[1]

hidden_edge_mask = [
    (src.item() in hidden_cve_ids) or (dst.item() in hidden_cve_ids)
    for src, dst in zip(src_nodes_all, dst_nodes_all)
]

hidden_edge_mask = torch.tensor(hidden_edge_mask)

print(hidden_edge_mask.shape)

torch.Size([15573])


### Count Hidden Edges

In [14]:
print("Hidden edges:", hidden_edge_mask.sum().item())
print("Remaining (train) edges:", (~hidden_edge_mask).sum().item())

Hidden edges: 921
Remaining (train) edges: 14652


### Create Train Graph (hidden CVE edges removed)

In [15]:
train_edge_index = edge_index[:, ~hidden_edge_mask]
train_edge_type = edge_type[~hidden_edge_mask]

print(train_edge_index.shape)
print(train_edge_type.shape)

torch.Size([2, 14652])
torch.Size([14652])


### Create Zero-Shot Test Graph (never used in training)

In [16]:
zero_shot_edge_index = edge_index[:, hidden_edge_mask]
zero_shot_edge_type = edge_type[hidden_edge_mask]

print(zero_shot_edge_index.shape)
print(zero_shot_edge_type.shape)

torch.Size([2, 921])
torch.Size([921])


In [17]:
print(train_edge_index.shape[1] + zero_shot_edge_index.shape[1])
print(edge_index.shape[1])

15573
15573


### Build Transformer Layer


In [4]:
class ThreatTransformer(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
        num_heads=4,
        num_layers=2
    ):

        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        return self.transformer(x)

### Initialize Model

In [18]:
tx_model = ThreatTransformer(
    embedding_dim=128,
    num_heads=4,
    num_layers=2
)

tx_model

ThreatTransformer(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

### Prepare Embeddings

In [19]:
emb_input = node_embeddings.unsqueeze(0)

print(emb_input.shape)

torch.Size([1, 7502, 128])


### Initial Forward Pass

In [20]:
# Initial forward pass — gradients are ON so the transformer trains end-to-end
tx_embeddings = tx_model(emb_input)

print(tx_embeddings.shape)

torch.Size([1, 7502, 128])


### Remove Batch Dimension

In [21]:
tx_embeddings = tx_embeddings.squeeze(0)

print(tx_embeddings.shape)

torch.Size([7502, 128])


In [22]:
print(tx_embeddings.shape)

torch.Size([7502, 128])


### Tx Link Predictor

In [23]:
class TxLinkPredictor(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            embedding_dim * 2,
            128
        )

        self.fc2 = nn.Linear(
            128,
            64
        )

        self.fc3 = nn.Linear(
            64,
            1
        )

    def forward(
        self,
        src,
        dst
    ):

        x = torch.cat(
            [src, dst],
            dim=1
        )

        x = F.relu(
            self.fc1(x)
        )

        x = F.relu(
            self.fc2(x)
        )

        x = self.fc3(x)

        return x

### Initialize

In [24]:
tx_predictor = TxLinkPredictor()

tx_predictor

TxLinkPredictor(
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)

### Build Positive Edges (train split only)


In [25]:
train_src_nodes = train_edge_index[0]
train_dst_nodes = train_edge_index[1]

print(train_src_nodes.shape)
print(train_dst_nodes.shape)

torch.Size([14652])
torch.Size([14652])


### Build Negative Edges

In [26]:
num_train_edges = train_edge_index.shape[1]

negative_src = torch.randint(0, num_nodes, (num_train_edges,))
negative_dst = torch.randint(0, num_nodes, (num_train_edges,))

print(negative_src.shape)

torch.Size([14652])


### Create Training Samples

In [27]:
positive_labels = torch.ones(num_train_edges)
negative_labels = torch.zeros(num_train_edges)

all_src = torch.cat([train_src_nodes, negative_src])
all_dst = torch.cat([train_dst_nodes, negative_dst])
all_labels = torch.cat([positive_labels, negative_labels])

print(all_src.shape)
print(all_labels.shape)

torch.Size([29304])
torch.Size([29304])


### Train/Val Split (within the train-edge pool only)

In [28]:
indices = torch.arange(len(all_labels))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.3,
    random_state=42
)

print(len(train_idx))
print(len(val_idx))

20512
8792


### Optimizer

In [29]:
import torch.optim as optim

# Re-initialise both models so we start fresh
tx_model = ThreatTransformer(
    embedding_dim=128,
    num_heads=4,
    num_layers=2
)

tx_predictor = TxLinkPredictor()

# Joint optimizer — trains transformer + predictor together end-to-end
optimizer = optim.Adam(
    list(tx_model.parameters()) +
    list(tx_predictor.parameters()),
    lr=0.001
)

criterion = torch.nn.BCEWithLogitsLoss()



### Train

In [31]:
epochs = 20

# node_embeddings from R-GCN checkpoint — used as input to transformer
emb_input = node_embeddings.unsqueeze(0)  # (1, num_nodes, 128)

for epoch in range(epochs):

    tx_model.train()
    tx_predictor.train()

    tx_out = tx_model(emb_input)      # (1, num_nodes, 128)
    tx_out = tx_out.squeeze(0)        # (num_nodes, 128)

    src_embed = tx_out[all_src[train_idx]]
    dst_embed = tx_out[all_dst[train_idx]]
    labels = all_labels[train_idx].unsqueeze(1)

    predictions = tx_predictor(src_embed, dst_embed)

    loss = criterion(predictions, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

Epoch 1 Loss: 0.6967
Epoch 2 Loss: 0.6285
Epoch 3 Loss: 0.5634
Epoch 4 Loss: 0.5002
Epoch 5 Loss: 0.4504
Epoch 6 Loss: 0.4061
Epoch 7 Loss: 0.3692
Epoch 8 Loss: 0.3373
Epoch 9 Loss: 0.3062
Epoch 10 Loss: 0.2780
Epoch 11 Loss: 0.2595
Epoch 12 Loss: 0.2430
Epoch 13 Loss: 0.2354
Epoch 14 Loss: 0.2197
Epoch 15 Loss: 0.2042
Epoch 16 Loss: 0.1983
Epoch 17 Loss: 0.2028
Epoch 18 Loss: 0.2108
Epoch 19 Loss: 0.1838
Epoch 20 Loss: 0.2082


### In-Distribution Validation (held-out edges from the train pool, not zero-shot)

In [32]:
tx_model.eval()
tx_predictor.eval()

with torch.no_grad():

    tx_embeddings = tx_model(emb_input).squeeze(0)

    src_embed = tx_embeddings[all_src[val_idx]]
    dst_embed = tx_embeddings[all_dst[val_idx]]

    logits = tx_predictor(src_embed, dst_embed)
    probs = torch.sigmoid(logits).squeeze()
    preds = (probs > 0.5).int()

    y_true = all_labels[val_idx].numpy()
    y_pred = preds.numpy()
    y_prob = probs.numpy()

print("Validation Accuracy :", accuracy_score(y_true, y_pred))
print("Validation Precision:", precision_score(y_true, y_pred))
print("Validation Recall   :", recall_score(y_true, y_pred))
print("Validation F1 Score :", f1_score(y_true, y_pred))
print("Validation AUC      :", roc_auc_score(y_true, y_prob))

Validation Accuracy : 0.9261828935395814
Validation Precision: 0.8761274804570054
Validation Recall   : 0.9929577464788732
Validation F1 Score : 0.9308912788840379
Validation AUC      : 0.9711971623997789


### Zero-Shot Evaluation Dataset (the held-out CVE edges)

In [33]:
zs_src = zero_shot_edge_index[0]
zs_dst = zero_shot_edge_index[1]

print(len(zs_src))

921


### Create Negative Samples for the Zero-Shot Set

In [34]:
num_zs = len(zs_src)

zs_negative_src = torch.randint(0, num_nodes, (num_zs,))
zs_negative_dst = torch.randint(0, num_nodes, (num_zs,))

print(len(zs_negative_src))

921


### Build Zero-Shot Test Set

In [36]:
zs_all_src = torch.cat([zs_src, zs_negative_src])
zs_all_dst = torch.cat([zs_dst, zs_negative_dst])

zs_labels = torch.cat([
    torch.ones(num_zs),
    torch.zeros(num_zs)
])

print(zs_all_src.shape)
print(zs_labels.shape)

torch.Size([1842])
torch.Size([1842])


In [37]:
print(train_edge_index.shape)
print(zero_shot_edge_index.shape)
print(zs_all_src.shape)
print(zs_labels.shape)

torch.Size([2, 14652])
torch.Size([2, 921])
torch.Size([1842])
torch.Size([1842])


### Run Strict Zero-Shot Predictions

In [38]:
tx_model.eval()
tx_predictor.eval()

with torch.no_grad():

    tx_embeddings = tx_model(node_embeddings.unsqueeze(0)).squeeze(0)

    src_embed = tx_embeddings[zs_all_src]
    dst_embed = tx_embeddings[zs_all_dst]

    logits = tx_predictor(src_embed, dst_embed)
    probs = torch.sigmoid(logits).squeeze()
    preds = (probs > 0.5).int()

### Compute Strict Zero-Shot Metrics

In [39]:
y_true = zs_labels.numpy()
y_pred = preds.numpy()
y_prob = probs.numpy()

print("Tx Strict Zero-Shot Accuracy :", accuracy_score(y_true, y_pred))
print("Tx Strict Zero-Shot Precision:", precision_score(y_true, y_pred))
print("Tx Strict Zero-Shot Recall   :", recall_score(y_true, y_pred))
print("Tx Strict Zero-Shot F1 Score :", f1_score(y_true, y_pred))
print("Tx Strict Zero-Shot AUC      :", roc_auc_score(y_true, y_prob))

Tx Strict Zero-Shot Accuracy : 0.9229098805646037
Tx Strict Zero-Shot Precision: 0.8777885548011639
Tx Strict Zero-Shot Recall   : 0.9826275787187839
Tx Strict Zero-Shot F1 Score : 0.9272540983606558
Tx Strict Zero-Shot AUC      : 0.9560502262918205
